# Model Development, Training, and Hyperparameter Optimization

## 🔗 Builds on (AIAT 111–125)

AIAT 126 is the *graduation* project: it **assembles** what you already built across the diploma, it does not re-teach it. Open what you need:

- **AIAT 114 U5 — grid search, the tuning loop this notebook runs on your data** → [`Course 04/unit5-model-selection/examples/01_grid_search.ipynb`](../../../Course%2004/unit5-model-selection/examples/01_grid_search.ipynb)
- **AIAT 111 U3 — gradient descent and loss functions: whatever architecture you picked, it still learns the same way** → [`Course 01/unit3-ml-basics/examples/04_gradient_descent_loss_functions.ipynb`](../../../Course%2001/unit3-ml-basics/examples/04_gradient_descent_loss_functions.ipynb)
- **AIAT 114 U2 — cross-validation and learning curves** → [`Course 04/unit2-regression-model-evaluation/examples/01_cross_validation.ipynb`](../../../Course%2004/unit2-regression-model-evaluation/examples/01_cross_validation.ipynb)
- **AIAT 113 U3 — optimizers and loss functions, when tuning stops helping** → [`Course 03/unit3-optimization/examples/01_optimizers_comparison.ipynb`](../../../Course%2003/unit3-optimization/examples/01_optimizers_comparison.ipynb)
- **AIAT 122 U1 — only if your model is a neural network** → [`Course 08/unit1-deep-learning-basics/examples/06_optimization_techniques.ipynb`](../../../Course%2008/unit1-deep-learning-basics/examples/06_optimization_techniques.ipynb)
- **AIAT 125 U5 — experiment tracking, so your tuning runs are reproducible** → [`Course 11/unit5-pipelines-monitoring/examples/05_experiment_tracking_mlflow_wandb.ipynb`](../../../Course%2011/unit5-pipelines-monitoring/examples/05_experiment_tracking_mlflow_wandb.ipynb)

Record every artifact you reuse in **section 6, the Prior Work Inventory**, of [`TEMPLATES/project_proposal_template.md`](../../TEMPLATES/project_proposal_template.md). It is scored at gate 1 (criterion 1.7) and re-checked at the Unit 2 design review — it is this course's **CLO2** evidence, and the official AIAT 126 description requires it.

## 📚 Learning Objectives

By completing this notebook, you will:
- Implement model architecture and training pipeline
- Train models with different hyperparameter configurations
- Perform hyperparameter optimization using grid search and automated tools
- Evaluate model performance using appropriate metrics
- Read tuning results against the baseline honestly and decide when refinement is worth it

## 🔗 Where this leads

**Used later in:** Course 12 — Unit 4, which evaluates the tuned model and decides whether the tuning was worth it.

---

This notebook covers practical activities from **Course 12, Unit 3**:
- Implementing model architecture and training pipeline
- Training models with different hyperparameter configurations
- Performing hyperparameter optimization using grid search or automated tools
- Evaluating model performance using appropriate metrics
- Analyzing model outputs and identifying areas for improvement
- Interpreting tuning results against the baseline honestly
- Documenting training procedures and results

---

## Introduction

**Model Development** involves selecting appropriate algorithms, designing architectures, training models, and optimizing hyperparameters. Tuning is a disciplined search — as this notebook's own results will show, it is *not* a guarantee of beating a strong default baseline, and reading its results honestly is part of the craft.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- **Your own project dataset**, if you have one: put it at `data/my_project_data.csv` next to this
  notebook (same convention as notebook 01 of this unit) and this pipeline will pick it up
- Otherwise a real named fallback — the 1912 Titanic passenger manifest at
  `../../../Course 04/datasets/raw/titanic.csv` — so the tuning runs against a genuine signal ceiling

**Outputs:** What you'll see when you run the cells

- Baseline metrics, a grid search and a random search over Random Forest hyperparameters,
  and a computed verdict on whether the tuning was actually worth it.

---

---

## The $1,000,000 of tuning that was never deployed

In October 2006 Netflix offered **$1 million** to anyone who could improve its rating predictions by 10%. Three years and thousands of teams later, "BellKor's Pragmatic Chaos" cleared the bar and took the cheque. The winning entry was an ensemble of **more than 800 models** blended together.

Netflix never put it into production. Its engineering blog gave the reason in 2012, in one sentence: *"we evaluated some of the new methods offline but the additional accuracy gains that we measured did not seem to justify the engineering effort needed to bring them into a production environment."* Two components of the solution — an SVD and a restricted Boltzmann machine — went into the product. The other several hundred did not. And by the time the prize was awarded Netflix had become a streaming company, so the thing the competition had spent three years optimising — predicting a five-star rating for a DVD arriving in three days — was no longer the thing the business needed.

Every element of that story is available to you in this notebook, at a scale of minutes instead of years: a search that optimises a number, a number that improves, and a separate question the search itself can never answer — **was it worth it?**

### What goes wrong without this

The failure this lesson prevents is not "my model is under-tuned". It is **spending the last three weeks of a capstone on hyperparameters** and arriving at the defence with a model 0.3 points better than the default, no baseline comparison, no failure analysis and no working demo. Tuning is the most seductive activity in a machine-learning project because it always produces a number and never produces a decision.

It also fails quietly in a specific way: you tune, you check the validation set, you tune again, you check again — and after enough rounds your validation score has stopped measuring the model and started measuring how many times you looked.

### The same reasoning, two other fields

- **Neural recommenders — Ferrari Dacrema, Cremonesi & Jannach (RecSys 2019).** The authors took **18** neural recommendation methods published at top conferences. Only **7** could be reproduced at all, and **6 of those 7** were beaten by simple, well-tuned baselines such as nearest-neighbour methods; the seventh did not consistently beat a well-tuned linear ranking method. Much of the published "progress" was a tuning-budget gap between the new method and the baseline it was measured against.
- **Neural language models — Melis, Dyer & Blunsom (ICLR 2018).** Given a large automatic hyperparameter-search budget, a *plain* LSTM outperformed the newer architectures published after it, setting new state-of-the-art results on Penn Treebank and WikiText-2. Again: what looked like architectural progress was partly a difference in how hard each model had been tuned.

Read those next to the cells you are about to run, where a default random forest and a tuned one finish within noise of each other. Two research literatures found the same thing at scale: **an untuned baseline is not a fair baseline — and a tuned baseline is often all you needed.**

In [1]:
# Import everything used below: data generation, model, metrics, and the three
# search/validation tools (train_test_split, GridSearchCV/RandomizedSearchCV, cross_val_score).
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("✅ Libraries imported!")


✅ Libraries imported!


In [2]:
# Load the project dataset: YOUR file if it exists, a real named fallback otherwise.
# WHY a real dataset matters here: hyperparameter tuning is only interesting when there is
# a genuine ceiling to bump into. On invented data you can usually reach any score you like
# by making the problem easier, which teaches you nothing about tuning.
PROJECT_DATA = Path("data/my_project_data.csv")      # <- your own dataset goes here
FALLBACK_DATA = Path("../../../Course 04/datasets/raw/titanic.csv")

if PROJECT_DATA.exists():
    raw_data = pd.read_csv(PROJECT_DATA)
    data_source = f"YOUR project dataset ({PROJECT_DATA})"
    # EDIT: name your own target column and drop any identifier columns.
    TARGET = "target"
    features = raw_data.drop(columns=[TARGET])
    X = pd.get_dummies(features, drop_first=True).fillna(0).to_numpy(dtype=float)
    y = raw_data[TARGET].to_numpy()
else:
    raw_data = pd.read_csv(FALLBACK_DATA)
    data_source = f"fallback: real Titanic passenger manifest ({FALLBACK_DATA.name}, 1912)"
    print(f"ℹ️  No file at {PROJECT_DATA} — using the real fallback dataset instead.\n")

    # Real gaps need real decisions before any model can be fitted.
    clean = raw_data.copy()
    clean["Age"] = clean["Age"].fillna(clean["Age"].median())
    clean["Embarked"] = clean["Embarked"].fillna(clean["Embarked"].mode()[0])
    features = pd.DataFrame({
        "pclass": clean["Pclass"],
        "is_female": (clean["Sex"] == "female").astype(int),
        "age": clean["Age"],
        "sibsp": clean["SibSp"],
        "parch": clean["Parch"],
        "fare": clean["Fare"],
        "family_size": clean["SibSp"] + clean["Parch"] + 1,
        "embarked_C": (clean["Embarked"] == "C").astype(int),
        "embarked_Q": (clean["Embarked"] == "Q").astype(int),
    })
    TARGET = "Survived"
    X = features.to_numpy(dtype=float)
    y = clean[TARGET].to_numpy()

# Hold out 20% as a validation set; stratify keeps the class balance in both parts
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Source: {data_source}")
print(f"Features ({X.shape[1]}): {list(features.columns)}")
print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}")
print(f"Class balance overall: {np.bincount(y).tolist()}")
print(f"Majority-class baseline on the validation set: {max(np.bincount(y_val)) / len(y_val):.4f}")

ℹ️  No file at data/my_project_data.csv — using the real fallback dataset instead.

Source: fallback: real Titanic passenger manifest (titanic.csv, 1912)
Features (9): ['pclass', 'is_female', 'age', 'sibsp', 'parch', 'fare', 'family_size', 'embarked_C', 'embarked_Q']
Training set: (712, 9), Validation set: (179, 9)
Class balance overall: [549, 342]
Majority-class baseline on the validation set: 0.6145


## Part 1: Baseline Model Training

Train a baseline model to establish performance benchmarks.


In [3]:
# Baseline first: train a random forest with DEFAULT hyperparameters — every
# tuning result later is judged against these numbers, so without a baseline
# you cannot tell whether tuning helped at all.
# Fit the out-of-the-box model on the training set
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

# Score the baseline on the held-out validation set with the four headline metrics
y_val_pred = baseline_model.predict(X_val)
baseline_accuracy = accuracy_score(y_val, y_val_pred)
baseline_precision = precision_score(y_val, y_val_pred)
baseline_recall = recall_score(y_val, y_val_pred)
baseline_f1 = f1_score(y_val, y_val_pred)

print("=" * 60)
print("Baseline Model Performance")
print("=" * 60)
print(f"Accuracy:  {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall:    {baseline_recall:.4f}")
print(f"F1-Score:  {baseline_f1:.4f}")
print(f"(Majority-class baseline accuracy: {max(np.bincount(y_val)) / len(y_val):.4f})")

# Per-class precision/recall breakdown, kept for the project record
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))


Baseline Model Performance
Accuracy:  0.8045
Precision: 0.7656
Recall:    0.7101
F1-Score:  0.7368
(Majority-class baseline accuracy: 0.6145)

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       110
           1       0.77      0.71      0.74        69

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



## Part 2: Hyperparameter Optimization - Grid Search


In [4]:
# Grid search: exhaustively try every combination in a small parameter grid,
# scored by cross-validated F1 on the TRAINING folds (the validation set stays out).
# Define hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Grid Search
print("=" * 60)
print("Grid Search Hyperparameter Optimization")
print("=" * 60)
print("Searching over parameter combinations...")
print(f"Total combinations: {np.prod([len(v) for v in param_grid.values()])}")

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42), param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Evaluate best model
best_model = grid_search.best_estimator_
y_val_pred_best = best_model.predict(X_val)
best_accuracy = accuracy_score(y_val, y_val_pred_best)

print(f"\nValidation Accuracy with best params: {best_accuracy:.4f}")
# Signed change vs the baseline — watch the sign, it can be negative!
print(f"Change vs baseline accuracy: {best_accuracy - baseline_accuracy:+.4f}")


Grid Search Hyperparameter Optimization
Searching over parameter combinations...
Total combinations: 108
Fitting 5 folds for each of 108 candidates, totalling 540 fits



Best parameters: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 50}
Best cross-validation score: 0.7629

Validation Accuracy with best params: 0.7933
Change vs baseline accuracy: -0.0112


### Reading the result honestly: tuning did not beat the baseline here

Look at the printed numbers above: the tuned model's validation accuracy came
out *lower* than the baseline's (0.7933 vs 0.8045). That is a real and common
outcome — not a bug. Three things explain it:

1. **The search optimized a different criterion.** Grid search picked the
   parameters with the best *cross-validated F1 on the training folds* — it
   never saw the validation set. Judged by its own criterion, the search did
   its job (the next cell verifies this).
2. **The validation set is small.** With 179 passengers, one flipped prediction
   moves accuracy by 0.56%, so a gap of about one point is two predictions wide.
3. **Real data has a ceiling.** Survival on the Titanic was never a clean
   function of class, sex, age and fare, so no hyperparameter setting was going
   to push this much past 0.80. When the defaults are already near the ceiling,
   tuning has almost nothing left to win.

The professional response — the one to practice for your capstone — is to
report the result as it is, check the criterion the search actually optimized,
and resist re-tuning until the validation set says what you want: that is how
data leakage starts.

In [5]:
# Verify the search on ITS OWN criterion: compare cross-validated F1 of the
# default baseline vs the tuned model — the metric grid search optimized.
baseline_cv_f1 = cross_val_score(
    RandomForestClassifier(random_state=42), X_train, y_train, cv=5, scoring='f1'
).mean()

print('=' * 60)
print("Baseline vs Tuned — on the search's own criterion (CV F1)")
print('=' * 60)
print(f'Baseline (default params) CV F1: {baseline_cv_f1:.4f}')
print(f'Tuned (best params) CV F1:       {grid_search.best_score_:.4f}')

# Compute the verdict from the numbers instead of asserting it in prose
cv_delta = grid_search.best_score_ - baseline_cv_f1
val_delta = best_accuracy - baseline_accuracy
print(f'\nCV F1 change from tuning:   {cv_delta:+.4f}')
print(f'Validation accuracy change: {val_delta:+.4f}')

# Branch on what the numbers actually say, so the conclusion is computed, not asserted
if cv_delta >= 0 and val_delta < 0:
    print('\nInterpretation: tuning improved (or matched) the cross-validated score')
    print(f'it optimized, but did NOT improve accuracy on the {len(y_val)}-sample validation')
    print('set. With defaults this strong, honest reporting beats chasing noise.')
elif cv_delta >= 0:
    print('\nInterpretation: tuning improved both the CV score and validation accuracy.')
else:
    print('\nInterpretation: tuning failed to improve even its own CV criterion —')
    print('widen or rethink the search space before drawing conclusions.')

Baseline vs Tuned — on the search's own criterion (CV F1)
Baseline (default params) CV F1: 0.7347
Tuned (best params) CV F1:       0.7629

CV F1 change from tuning:   +0.0282
Validation accuracy change: -0.0112

Interpretation: tuning improved (or matched) the cross-validated score
it optimized, but did NOT improve accuracy on the 179-sample validation
set. With defaults this strong, honest reporting beats chasing noise.


## Part 3: Hyperparameter Optimization - Random Search

Random search is more efficient for large parameter spaces.


In [6]:
# Random search: instead of trying EVERY combination like grid search, sample
# 20 random ones — far cheaper in large spaces, and usually nearly as good
# because only a few hyperparameters really matter (Bergstra & Bengio, 2012).
# A slightly wider space than the grid — random search can afford it
param_distributions = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6]
}

print("=" * 60)
print("Random Search Hyperparameter Optimization")
print("=" * 60)
print("Searching random parameter combinations (more efficient for large spaces)...")

# Same 5-fold CV and F1 criterion as grid search, but only 20 sampled configs
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), param_distributions,
    n_iter=20,  # Try 20 random combinations
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best cross-validation score: {random_search.best_score_:.4f}")

# Compare with grid search on the SAME criterion: cross-validated F1
print(f"\nGrid Search best score: {grid_search.best_score_:.4f}")
print(f"Random Search best score: {random_search.best_score_:.4f}")
print(f"Difference: {abs(grid_search.best_score_ - random_search.best_score_):.4f}")


Random Search Hyperparameter Optimization
Searching random parameter combinations (more efficient for large spaces)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits



Best parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 20}
Best cross-validation score: 0.7555

Grid Search best score: 0.7629
Random Search best score: 0.7555
Difference: 0.0074


### What the two searches cost, and what they bought

Put the printed numbers side by side:

| Search | Fits performed | Best CV F1 |
|---|---|---|
| Grid — exhaustive | 108 combinations × 5 folds = **540 fits** | **0.7629** |
| Random — 20 samples of a *wider* space | 20 × 5 folds = **100 fits** | **0.7555** |

Random search used **under a fifth of the compute** and finished **0.0074** behind. A gap that size between two cross-validated estimates on 712 training rows is not a meaningful difference — change the seed and it could reverse. This is exactly what Bergstra and Bengio (2012) reported: in most spaces only a couple of hyperparameters matter much, so sampling finds the good region about as fast as enumerating it, while enumeration's cost multiplies with every parameter you add. **Add one more four-value parameter to the grid and it becomes 432 combinations, 2,160 fits; random search still costs 100.**

And note what the exhaustive search bought against the *baseline*: **+0.0282** CV F1 for 540 fits, and **−0.0112** accuracy on the validation set. Both numbers belong in your report. A tuning section that reports only the improvement it likes is precisely the habit gate 4 exists to catch.

## 💬 Discuss

Three questions with no single right answer. Argue from the numbers this notebook printed.

1. **Which number goes on your results slide?** This run produced three true headlines: *"tuning improved cross-validated F1 by 0.0282"*; *"tuning reduced validation accuracy by 0.0112"*; and *"a random forest with default settings beats the majority-class baseline by nineteen points"* (0.8045 against 0.6145). All three are honest. Which do you show, and what does your choice say about what you believe the project is for? What would an examiner accuse you of if you showed only the first?
2. **When do you stop?** You have 640 fits behind you and a model that is not better than the default. Argue for exactly one of: widen the search space; change the model family; go and build new features; declare the default final and spend the remaining days on evaluation. Ground the argument in something printed above — the nine feature names, the 0.6145 majority baseline, the ~0.80 ceiling — not in a preference.
3. **Would you pay for this?** Netflix decided that several hundred models' worth of extra accuracy was not worth the engineering. Put a price on yours: if every 0.01 of F1 cost one extra day of the project, how many days is your current gain worth? Then answer the version a client asks instead: **what does 0.01 of F1 actually do for the person using the system?** If you cannot answer that second question, what does it tell you about the metric you chose?

## ⚠️ Where this breaks

Hyperparameter search has the widest gap in this course between how productive it feels and how much it usually earns. Here is where it stops paying.

- **A search can only find what you put in the grid.** `max_depth` ran over `[5, 10, 20, None]` and the winner was 20 — an interior value, so the space was wide enough this time. Had the winner sat at an edge of a list, the honest conclusion would be *"the search hit the boundary, so the grid was wrong"*, not *"20 is best"*. Check where the winner sits in every list before you report it.
- **Tuning cannot repair a feature set.** Both searches flatten out near 0.80 accuracy because that is roughly what these nine columns support — whether a 1912 passenger survived was never a clean function of class, sex, age, fare and family size. **The assumption that has to hold before tuning is worth doing: the features already carry the signal, and only the model's flexibility is holding you back.** When that is false, one hour spent building a new feature beats a day of grid search every time.
- **Tuning against a small validation set mostly measures the validation set.** With 179 rows, one flipped prediction is 0.56% of accuracy. That is why the tuned model came out 0.0112 *below* the baseline on validation while sitting 0.0282 *above* it on cross-validated F1: the two disagree because one of them is barely distinguishable from noise. Choose with cross-validation, keep the held-out set for one final measurement, and do not re-tune after looking at it — every extra look converts a held-out set a little further into a training set.
- **`scoring='f1'` was a decision, and every result above inherits it.** Set the scorer to `recall` and the search returns different winners, because it is optimising a different definition of good. Pick the metric from the harm you are trying to avoid, in the proposal, *before* the search — otherwise you will end up picking the metric that flatters the model you already have.
- **One random seed is one sample.** Every number here came from `random_state=42`. Re-run with three other seeds before claiming any difference under roughly two points is real. Reporting a single seed as though it were a result is the most common statistical error in student capstones.
- **When NOT to tune at all.** If your model has not yet beaten a baseline someone could compute *without* machine learning, tuning is the wrong tool: you have a framing problem or a feature problem, not a hyperparameter problem. And when the defaults already sit near the ceiling — as they do here — the professional move is to write *"the tuned model was within noise of the defaults, so the defaults were kept"* and spend the days on evaluation, failure analysis and the demo. Those are worth 35 of the 100 points in this course. Hyperparameters are worth none of them directly.

## Summary

### Key Concepts:
1. **Baseline Model**: Establish performance benchmark with default parameters
2. **Grid Search**: Exhaustive search over all parameter combinations (computationally expensive)
3. **Random Search**: Sample random combinations (more efficient for large spaces)
4. **Cross-Validation**: Use CV scores for hyperparameter selection, not validation set
5. **Honest reporting**: judge a search on the criterion it optimized; a strong default baseline is a legitimate final choice

### What happened in this run (real Titanic manifest):
Grid search raised the cross-validated F1 it was optimizing from 0.7347 to 0.7629 (+0.0282), yet the tuned model's accuracy on the held-out 179 passengers came out at 0.7933 against the baseline's 0.8045 (−0.0112). Random search, sampling only 20 of a wider space, landed at CV F1 0.7555 — within 0.0074 of the exhaustive grid, for a fifth of the fits. Both searches comfortably beat the 0.6145 majority-class baseline, and neither pushed past roughly 0.80, which is about as far as these nine columns can carry a model. The "Reading the result honestly" section above shows how to interpret that.

### Best Practices:
- Always establish baseline first
- Use cross-validation for hyperparameter tuning
- Keep validation set separate for final evaluation
- Document all hyperparameter configurations tried
- Track training time vs performance trade-offs

### Running this on your own project data
Save your dataset as `data/my_project_data.csv`, set `TARGET` in the loading cell to your label
column, and re-run. Everything below adapts — including the computed verdict on whether tuning helped.

**Reference:** Course 12, Unit 3: "Implementation and Development of the Project Idea" — training and tuning activities demonstrated above; see notebook 01 of this unit for the data collection and preprocessing pipeline.


## 📚 References

1. Breiman, L. (2001). *Random Forests*. Machine Learning, 45(1), 5–32.
2. Bergstra, J., & Bengio, Y. (2012). *Random Search for Hyper-Parameter Optimization*. Journal of Machine Learning Research, 13, 281–305. <https://jmlr.org/papers/v13/bergstra12a.html>
3. Snoek, J., Larochelle, H., & Adams, R. P. (2012). *Practical Bayesian Optimization of Machine Learning Algorithms*. NeurIPS 2012. <https://arxiv.org/abs/1206.2944>
4. Bischl, B., Binder, M., Lang, M., et al. (2021). *Hyperparameter Optimization: Foundations, Algorithms, Best Practices and Open Challenges*. arXiv. <https://arxiv.org/abs/2107.05847>

**Cases cited in this lesson**

5. Ferrari Dacrema, M., Cremonesi, P., & Jannach, D. (2019). *Are We Really Making Much Progress? A Worrying Analysis of Recent Neural Recommendation Approaches*. **RecSys 2019.** <https://arxiv.org/abs/1907.06902> — 18 neural methods, 7 reproducible, 6 of those beaten by well-tuned simple baselines.
6. Melis, G., Dyer, C., & Blunsom, P. (2018). *On the State of the Art of Evaluation in Neural Language Models*. **ICLR 2018.** <https://arxiv.org/abs/1707.05589> — a properly regularised, heavily tuned standard LSTM outperforming the architectures published after it.
7. Amatriain, X., & Basilico, J. (2012). *Netflix Recommendations: Beyond the 5 Stars (Part 1)*. Netflix Technology Blog — the statement that the Netflix Prize winning ensemble was not deployed because the measured accuracy gains "did not seem to justify the engineering effort needed to bring them into a production environment."
